## Prefill- time increase with input len

In [51]:
import json
import matplotlib.pyplot as plt
import numpy as np

# Apply global plotting configurations
plt.rcParams['savefig.dpi'] = 600  # Image resolution
plt.rcParams['figure.dpi'] = 600   # Display resolution
plt.rcParams['xtick.major.width'] = 3
plt.rcParams['ytick.major.width'] = 3
plt.rcParams['grid.linestyle'] = '-'
plt.rcParams['grid.linewidth'] = 8
plt.rcParams['grid.color'] = '#e1e1e1'
plt.rcParams['axes.linewidth'] = 5
plt.rcParams['xtick.major.size'] = 10
plt.rcParams['ytick.major.size'] = 10
plt.rcParams['axes.labelsize'] = 65
plt.rcParams['lines.linewidth'] = 10
plt.rcParams['lines.markersize'] = 30
plt.rcParams['xtick.labelsize'] = 65
plt.rcParams['ytick.labelsize'] = 65
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['DejaVu Serif']
plt.rcParams['font.weight'] = 'normal'
plt.rcParams['axes.labelweight'] = 'normal'
plt.rcParams['axes.titleweight'] = 'normal'
plt.rcParams['figure.figsize'] = (17, 12)
# grid false
plt.rcParams['axes.grid'] = False
# set legend fontsize
plt.rcParams['legend.fontsize'] = 42

# set title fontsize
plt.rcParams['axes.titlesize'] = 65


files = ["deepseek_1.3B_prefill_result.json", "llama3_8b_prefill_result.json"]
model_names = ["DeepSeek-1.3B", "Llama3-8B"]

# 样式映射： (model_name, num_req) -> dict of style attributes
style_mapping = {
    ("Llama3-8B", 8):      {"color": "#316ba6", "marker": "^"},
    ("Llama3-8B", 16):     {"color": "#479236", "marker": "D"},
    ("DeepSeek-1.3B", 8):  {"color": "#316ba6", "marker": "o"},
    ("DeepSeek-1.3B", 16): {"color": "#479236", "marker": "s"},
}

line_styles = {
    "DeepSeek-1.3B": '-',
    "Llama3-8B": '--'
}

# Create a figure
plt.figure()

# Load and group all data first
all_data = {}
for file_idx, file_name in enumerate(files):
    model_name = model_names[file_idx]
    with open(file_name, 'r') as f:
        data = json.load(f)
    grouped_by_num_req = {}
    for entry in data:
        num_req = entry['num_req']
        if num_req not in grouped_by_num_req:
            grouped_by_num_req[num_req] = []
        grouped_by_num_req[num_req].append(entry)

    # Sort entries by input_len
    for num_req in grouped_by_num_req:
        grouped_by_num_req[num_req] = sorted(grouped_by_num_req[num_req], key=lambda x: x['input_len'])

    all_data[model_name] = grouped_by_num_req

# Plot in the exact order of style_mapping
for (model_name, num_req), style in style_mapping.items():
    if model_name not in all_data or num_req not in all_data[model_name]:
        continue  # Skip if data is missing

    entries = all_data[model_name][num_req]
    input_lens = [entry['input_len'] for entry in entries]
    times = [entry['time'] for entry in entries]

    color = style["color"]
    marker = style["marker"]
    linestyle = line_styles[model_name]

    plt.plot(input_lens, times, marker=marker, color=color, linestyle=linestyle,
             label=f'{model_name}, Batch Size={num_req}')

# Labels and saving
plt.xlabel('Input Length')
plt.ylabel('Latency (s)')
plt.legend()
plt.savefig('prefill_latency_vs_input_length.pdf', bbox_inches='tight', dpi=600, format='pdf')
plt.close()


## decode-time increase with output len

In [35]:
def remove_outliers(token_idxs, times, threshold_multiplier=3):
    if not times:
        return [], []
    median_time = np.median(times)
    outlier_threshold = threshold_multiplier * median_time
    clean_times = []
    clean_idxs = []
    for idx, time in zip(token_idxs, times):
        if time <= outlier_threshold:
            clean_times.append(time)
            clean_idxs.append(idx)
    return clean_idxs, clean_times


In [63]:
import json
import matplotlib.pyplot as plt
import numpy as np
import os

# Apply global plotting configurations
plt.rcParams['savefig.dpi'] = 600  # Image resolution
plt.rcParams['figure.dpi'] = 600   # Display resolution
plt.rcParams['xtick.major.width'] = 3
plt.rcParams['ytick.major.width'] = 3
plt.rcParams['grid.linestyle'] = '-'
plt.rcParams['grid.linewidth'] = 8
plt.rcParams['grid.color'] = '#e1e1e1'
plt.rcParams['axes.linewidth'] = 5
plt.rcParams['xtick.major.size'] = 10
plt.rcParams['ytick.major.size'] = 10
plt.rcParams['axes.labelsize'] = 65
plt.rcParams['lines.linewidth'] = 3
plt.rcParams['lines.markersize'] = 30
plt.rcParams['xtick.labelsize'] = 65
plt.rcParams['ytick.labelsize'] = 65
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['DejaVu Serif']
plt.rcParams['font.weight'] = 'normal'
plt.rcParams['axes.labelweight'] = 'normal'
plt.rcParams['axes.titleweight'] = 'normal'
plt.rcParams['figure.figsize'] = (17, 12)
# grid false
plt.rcParams['axes.grid'] = False
# set legend fontsize
plt.rcParams['legend.fontsize'] = 42

# set title fontsize
plt.rcParams['axes.titlesize'] = 65

# set markersize
plt.rcParams['lines.markersize'] = 5

# 文件与模型名
files = ["llama3_8b_decode_result_different_output_len.json", "deepseek_1.3B_decode_result_different_output_len.json"]
model_names = ["Llama-3 8B", "DeepSeek 1.3B"]

# 样式映射
style_mapping = {
    ("Llama-3 8B", 32):      {"color": "#85a9d6", "marker": "s", "linestyle": "-"},  # 浅蓝
    ("Llama-3 8B", 128):     {"color": "#316ba6", "marker": "^", "linestyle": "-"},  # 深蓝
    ("DeepSeek 1.3B", 32):   {"color": "#a7d9a0", "marker": "s", "linestyle": "-"},  # 浅绿
    ("DeepSeek 1.3B", 128):  {"color": "#479236", "marker": "^", "linestyle": "-"},  # 深绿
}


# 预处理函数：去除异常值
def remove_outliers(x, y, z_score_thresh=2.0):
    import numpy as np
    x, y = np.array(x), np.array(y)
    if len(y) == 0:
        return [], []
    mean = np.mean(y)
    std = np.std(y)
    mask = np.abs(y - mean) < z_score_thresh * std
    return x[mask].tolist(), y[mask].tolist()

# 创建图形
plt.figure()

# 遍历文件
for file_idx, file in enumerate(files):
    model_name = model_names[file_idx]

    with open(file, 'r') as f:
        data = json.load(f)

    grouped_by_num_req = {}
    for entry in data:
        num_req = entry['num_req']
        if num_req not in grouped_by_num_req:
            grouped_by_num_req[num_req] = []
        grouped_by_num_req[num_req].append(entry)

    for num_req, entries in sorted(grouped_by_num_req.items()):
        for entry in entries:
            if 'time' in entry and isinstance(entry['time'], list) and len(entry['time']) > 100:
                valid_entry = entry['time'][50:-50]

                if all(isinstance(item, dict) and 'token_idx' in item and 'time' in item for item in valid_entry):
                    token_idxs = [item['token_idx'] for item in valid_entry]
                    times = [item['time'] for item in valid_entry]

                    clean_idxs, clean_times = remove_outliers(token_idxs, times)

                    if clean_idxs and clean_times:
                        style = style_mapping.get((model_name, num_req), {
                            "color": "gray", "marker": "x", "linestyle": "--"
                        })

                        plt.plot(clean_idxs, clean_times,
                                 color=style["color"],
                                 linestyle=style["linestyle"],
                                 label=f'{model_name}, Batch Size={num_req}')

# 添加图例与标签
plt.xlabel('Token Index')
plt.ylabel('Latency (s)')

legend = plt.legend()
for line in legend.get_lines():
    line.set_linewidth(10)
plt.savefig('decode_latency_vs_token_idx.pdf', bbox_inches='tight', dpi=600, format='pdf')
plt.close()

In [64]:
import json
import matplotlib.pyplot as plt
import numpy as np
import os


# Apply global plotting configurations
plt.rcParams['savefig.dpi'] = 600  # Image resolution
plt.rcParams['figure.dpi'] = 600   # Display resolution
plt.rcParams['xtick.major.width'] = 3
plt.rcParams['ytick.major.width'] = 3
plt.rcParams['grid.linestyle'] = '-'
plt.rcParams['grid.linewidth'] = 8
plt.rcParams['grid.color'] = '#e1e1e1'
plt.rcParams['axes.linewidth'] = 5
plt.rcParams['xtick.major.size'] = 10
plt.rcParams['ytick.major.size'] = 10
plt.rcParams['axes.labelsize'] = 65
plt.rcParams['lines.linewidth'] = 10
plt.rcParams['lines.markersize'] = 30
plt.rcParams['xtick.labelsize'] = 65
plt.rcParams['ytick.labelsize'] = 65
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['DejaVu Serif']
plt.rcParams['font.weight'] = 'normal'
plt.rcParams['axes.labelweight'] = 'normal'
plt.rcParams['axes.titleweight'] = 'normal'
plt.rcParams['figure.figsize'] = (17, 12)
# grid false
plt.rcParams['axes.grid'] = False
# set legend fontsize
plt.rcParams['legend.fontsize'] = 42

# set title fontsize
plt.rcParams['axes.titlesize'] = 65



# Define files and model names
files = [
    "llama3_8b_decode_result_different_batch_size.json", 
    "deepseek_1.3B_decode_result_different_batch_size.json"
]
model_names = ["Llama-3 8B", "DeepSeek 1.3B"]

# 样式映射： (model_name, token_idx) -> dict of style attributes
style_mapping = {
    ("Llama-3 8B", 512):      {"color": "#316ba6", "marker": "^", "linestyle": "--"},
    ("Llama-3 8B", 1024):     {"color": "#479236", "marker": "D", "linestyle": "--"},
    ("DeepSeek 1.3B", 512):   {"color": "#316ba6", "marker": "o", "linestyle": "-"},
    ("DeepSeek 1.3B", 1024):  {"color": "#479236", "marker": "s", "linestyle": "-"},
}

# Token indices of interest and smoothing window
target_token_idxs = [512, 1024]
window_size = 10

plt.figure()

# Load all data
all_data = {}
for file_idx, file in enumerate(files):
    model_name = model_names[file_idx]
    with open(file, 'r') as f:
        data = json.load(f)
    # Group by batch size (num_req)
    grouped_by_num_req = {}
    for entry in data:
        num_req = entry['num_req']
        grouped_by_num_req.setdefault(num_req, []).append(entry)
    
    all_data[model_name] = grouped_by_num_req

# Plot using style_mapping order
for (model_name, token_idx), style in style_mapping.items():
    if model_name not in all_data:
        continue

    grouped_by_num_req = all_data[model_name]
    plot_data = {'num_reqs': [], 'times': []}

    for num_req, entries in sorted(grouped_by_num_req.items()):
        for entry in entries:
            times_near_target = [
                t['time'] for t in entry['time']
                if abs(t['token_idx'] - token_idx) <= window_size
            ]
            if times_near_target:
                median_time = np.median(times_near_target)
                plot_data['num_reqs'].append(num_req)
                plot_data['times'].append(median_time)

    # Sort before plotting
    sorted_data = sorted(zip(plot_data['num_reqs'], plot_data['times']))
    if sorted_data:
        sorted_num_reqs, sorted_times = zip(*sorted_data)
        plt.plot(sorted_num_reqs, sorted_times,
                 marker=style['marker'],
                 color=style['color'],
                 linestyle=style['linestyle'],
                 label=f'{model_name}, Token Idx={token_idx}')

plt.xlabel('Batch Size')
plt.ylabel('Median Latency (s)')
plt.legend()
# plt.grid(True)
plt.savefig('decode_latency_vs_batch_size.pdf', bbox_inches='tight', dpi=600, format='pdf')
plt.close()
